# OOP 2: Inheritance

## Table of Contents

1. [What is Inheritance?](#1)
2. [Defining Subclasses](#2)
3. [Method Overriding](#3)
4. [super()](#4)
5. [A Realistic Example: Preprocessing Hierarchy](#5)
6. [Exercise: Extend the Dataset Class](#6)

---
## 1. What is Inheritance? <a id="1"></a>

Inheritance lets a class (the **subclass**) reuse and extend the behaviour of another class (the **superclass**). The subclass *is a* more specific version of the superclass.

```python
class SubclassName(SuperclassName):
    ...
```

You've already used this without knowing it. Every sklearn estimator inherits from `BaseEstimator`. Every pandas `DataFrame` inherits from `NDFrame`. The whole standard library is an inheritance tree.

The two things inheritance gives you:
- **Free behaviour**: methods defined on the superclass are available on the subclass without rewriting them
- **Substitutability**: anywhere the superclass is expected, the subclass works too

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# A simple superclass
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self, food):
        return f"{self.name} is eating {food}"

    def speak(self):
        raise NotImplementedError("Subclass must implement speak()")


# Subclasses inherit eat() for free and provide their own speak()
class Dog(Animal):
    def speak(self):
        return f"{self.name} says Woof!"

class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"


dog = Dog("Frank")
cat = Cat("Luna")

print(dog.speak())
print(cat.speak())
print(dog.eat("kibble"))   # inherited from Animal, so no need to redefine it
print()
print(isinstance(dog, Dog))     # True
print(isinstance(dog, Animal))  # also True: dog IS AN Animal

---
## 2. Defining Subclasses <a id="2"></a>

A subclass can:
- Inherit everything from the superclass as-is
- Add new attributes and methods
- Override existing methods with new behaviour

The subclass `__init__` can extend the superclass `__init__` by calling it explicitly.

In [ ]:
class BasePreprocessor:
    def __init__(self):
        self._fitted = False

    def fit(self, X):
        raise NotImplementedError("Subclass must implement fit()")

    def transform(self, X):
        raise NotImplementedError("Subclass must implement transform()")

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def __repr__(self):
        status = "fitted" if self._fitted else "not fitted"
        return f"{type(self).__name__}({status})"


class StandardScaler(BasePreprocessor):
    def __init__(self):
        super().__init__()   # call BasePreprocessor.__init__() to set _fitted
        self._mean = None
        self._std = None

    def fit(self, X):
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")
        return (X - self._mean) / self._std


X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})

scaler = StandardScaler()
print(scaler)                      # uses __repr__ from BasePreprocessor
print(scaler.fit_transform(X).round(2))  # uses fit_transform from BasePreprocessor
print(scaler)

---
## 3. Method Overriding <a id="3"></a>

A subclass can **override** any method from the superclass by redefining it. The subclass version takes precedence.

In [ ]:
class MinMaxScaler(BasePreprocessor):
    def __init__(self):
        super().__init__()
        self._min = None
        self._range = None

    def fit(self, X):
        self._min = X.min()
        self._range = (X.max() - X.min()).clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")
        return (X - self._min) / self._range

    # Override __repr__ to show the range
    def __repr__(self):
        status = "fitted" if self._fitted else "not fitted"
        return f"MinMaxScaler(feature_range=[0,1], {status})"


X_train = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})
X_test  = pd.DataFrame({"tenure": [17, 1],            "charges": [80, 15]})

scaler = MinMaxScaler()
print(repr(scaler))
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)
print(X_tr.round(2))
print()
print(repr(scaler))
print()
# Test values should be outside [0,1] if they're outside the training range — that's correct
print("Test scaled (may exceed [0,1] for out-of-range values):")
print(X_te.round(2))

---
## 4. super() <a id="4"></a>

`super()` calls the method on the **parent class**. Use it when you want to *extend* the parent's behaviour rather than replace it entirely.

In [ ]:
class LoggingScaler(StandardScaler):
    def __init__(self, name="scaler"):
        super().__init__()   # sets up _fitted, _mean, _std from StandardScaler
        self.name = name

    def fit(self, X):
        print(f"[{self.name}] fitting on {X.shape[0]} rows, {X.shape[1]} features")
        super().fit(X)       # do the actual fitting from StandardScaler
        print(f"[{self.name}] fit complete")
        return self

    def transform(self, X):
        print(f"[{self.name}] transforming {X.shape[0]} rows")
        return super().transform(X)   # do the actual transform from StandardScaler


ls = LoggingScaler(name="churn_scaler")
X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})
result = ls.fit_transform(X)
print(result.round(2))

### The MRO — Method Resolution Order

When Python looks up a method, it follows the **MRO**, the order in which parent classes are searched. You can inspect it:

In [ ]:
print("LoggingScaler MRO:")
for cls in LoggingScaler.__mro__:
    print(f"  {cls}")

---
## 5. A Realistic Example: Preprocessing Hierarchy <a id="5"></a>

Let's build the hierarchy we'll use throughout the rest of the course.

In [ ]:
class BasePreprocessor:
    def __init__(self):
        self._fitted = False

    def fit(self, X):
        raise NotImplementedError

    def transform(self, X):
        raise NotImplementedError

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def __repr__(self):
        return f"{type(self).__name__}({'fitted' if self._fitted else 'not fitted'})"


class StandardScaler(BasePreprocessor):
    def __init__(self):
        super().__init__()
        self._mean = None
        self._std = None

    def fit(self, X):
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted: raise RuntimeError("fit() first.")
        return (X - self._mean) / self._std


class MinMaxScaler(BasePreprocessor):
    def __init__(self):
        super().__init__()
        self._min = None
        self._range = None

    def fit(self, X):
        self._min = X.min()
        self._range = (X.max() - X.min()).clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted: raise RuntimeError("fit() first.")
        return (X - self._min) / self._range


class RobustScaler(BasePreprocessor):
    def __init__(self):
        super().__init__()
        self._median = None
        self._iqr = None

    def fit(self, X):
        self._median = X.median()
        self._iqr = (X.quantile(0.75) - X.quantile(0.25)).clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted: raise RuntimeError("fit() first.")
        return (X - self._median) / self._iqr


# All three share the same interface — same call, different behaviour
np.random.seed(42)
X_train = pd.DataFrame({
    "tenure":  np.random.randint(1, 60, 100),
    "charges": np.append(np.random.uniform(20, 80, 95), [500, 600, 700, 800, 900]),
})

for ScalerClass in [StandardScaler, MinMaxScaler, RobustScaler]:
    result = ScalerClass().fit_transform(X_train)
    print(f"{ScalerClass.__name__:20s}  charges mean={result['charges'].mean():.2f}  std={result['charges'].std():.2f}")

---
## 6. Exercise: Extend the Dataset Class <a id="6"></a>

Take the `Dataset` class from notebook 01 and create a `StratifiedDataset` subclass that:

1. Overrides `split()` to always use `stratify=self.y` (already true in the base) **and** additionally stores the class balance of each split as `self.train_balance` and `self.test_balance`
2. Adds a `balance_report()` method that prints both balances side by side
3. Uses `super().split()` — don't copy the splitting logic

In [ ]:
class Dataset:
    def __init__(self, X, y, name="dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits: dict = {}

    def split(self, test_size=0.2, random_state=42):
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self): return self._splits["X_train"], self._splits["y_train"]
    @property
    def test(self):  return self._splits["X_test"],  self._splits["y_test"]

    def __repr__(self):
        return f"Dataset(name={self.name!r}, shape={self.X.shape})"


# Your subclass here
class StratifiedDataset(Dataset):
    pass


# --- Test ---
np.random.seed(42)
X = pd.DataFrame({"tenure": np.random.randint(1, 60, 200),
                  "charges": np.random.uniform(20, 100, 200)})
y = pd.Series((np.random.rand(200) > 0.7).astype(int), name="churn")

ds = StratifiedDataset(X, y, name="churn").split()
ds.balance_report()

Do you need a bit of help? Here you can find a possible implementation of the class.

> <details>
> <summary>Click here</summary>
>
> ```python
> class StratifiedDataset(Dataset):
>     def __init__(self, X, y, name="dataset"):
>         super().__init__(X, y, name)
>         self.train_balance = None
>         self.test_balance = None
>
>     def split(self, test_size=0.2, random_state=42):
>         super().split(test_size=test_size, random_state=random_state)
>         _, y_tr = self.train
>         _, y_te = self.test
>         self.train_balance = y_tr.value_counts(normalize=True)
>         self.test_balance  = y_te.value_counts(normalize=True)
>         return self
>
>     def balance_report(self) -> None:
>         if self.train_balance is None:
>             raise RuntimeError("Call split() first.")
>         report = pd.DataFrame({
>             "train": self.train_balance,
>             "test":  self.test_balance,
>         }).round(3)
>         print(report)
> ```
> </details>